In [1]:
# gen_data.ipynb - generate a new dataset by subset from previous simulated dataset.

In [2]:
import numpy as np
import os
import pandas as pd
from sccnasim.xlib.xdata import load_10x_data, save_10x_data

In [3]:
in_cell_anno_fn = '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/cna_prediction/1n1t/base/gen_data/simu/4_rs/rs.cell_anno.tsv'
in_matrix_dir = '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/cna_prediction/1n1t/base/gen_data/rdr_starsolo'
out_matrix_dir = '/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/cna_prediction/1n1t_s1600/base/data/matrix'
out_cell_anno_fn = os.path.join(out_matrix_dir, 'spot_anno.tsv')

n_normal = 600
n_tumor = 1000
seed = n_normal

In [4]:
os.makedirs(out_matrix_dir, exist_ok = True)

# Load data

In [5]:
cell_anno = pd.read_csv(in_cell_anno_fn, sep = '\t', header = None)
cell_anno.columns = ['cell', 'clone']
cell_anno

,cell,clone
0,AAACCTGAGACCACGA-1,normal
1,AAACCTGCAAGTAGTA-1,normal
2,AAACCTGCAATAGAGT-1,normal
3,AAACCTGGTGTGAATA-1,normal
4,AAACCTGGTTTGCATG-1,normal
...,...,...
3595,TTTGTCACATCGTCGG-1,tumor
3596,TTTGTCAGTAGGCTGA-1,tumor
3597,TTTGTCAGTCATCCCT-1,tumor
3598,TTTGTCAGTGGGTCAA-1,tumor


In [6]:
cell_anno['clone'].value_counts()

clone
normal    1800
tumor     1800
Name: count, dtype: int64

In [7]:
adata = load_10x_data(data_dir = in_matrix_dir)
adata

/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 3600 × 33538
    obs: 'cell'
    var: 'feature_id', 'feature_name'

In [8]:
assert np.all(adata.obs['cell'].to_numpy() == cell_anno['cell'].to_numpy())

# Subset

### Subset cells

In [9]:
normal_cells = cell_anno.loc[cell_anno['clone'] == 'normal', 'cell'].to_numpy()
tumor_cells = cell_anno.loc[cell_anno['clone'] == 'tumor', 'cell'].to_numpy()
print(normal_cells.shape, tumor_cells.shape)

(1800,) (1800,)


In [10]:
np.random.seed(seed)
normal_cells_ds = np.random.choice(normal_cells, size = n_normal, replace = False)
print(normal_cells_ds.shape)

(600,)


In [11]:
tumor_cells_ds = np.random.choice(tumor_cells, size = n_tumor, replace = False)
print(tumor_cells_ds.shape)

(1000,)


In [12]:
new_cells = np.concatenate([normal_cells_ds, tumor_cells_ds])
print(new_cells.shape)

(1600,)


In [13]:
new_cell_anno = cell_anno[cell_anno['cell'].isin(new_cells)].copy()
new_cell_anno

,cell,clone
0,AAACCTGAGACCACGA-1,normal
1,AAACCTGCAAGTAGTA-1,normal
5,AAACCTGTCAGCGATT-1,normal
9,AAACGGGAGGTAGCTG-1,normal
11,AAACGGGGTATTAGCC-1,normal
...,...,...
3588,TTTGGTTGTGCTCTTC-1,tumor
3591,TTTGTCAAGGCTAGAC-1,tumor
3594,TTTGTCACACAGCCCA-1,tumor
3595,TTTGTCACATCGTCGG-1,tumor


In [14]:
assert len(np.unique(new_cell_anno['cell'])) == new_cell_anno.shape[0]

In [15]:
new_cell_anno['clone'].value_counts()

clone
tumor     1000
normal     600
Name: count, dtype: int64

In [16]:
new_cell_anno.to_csv(out_cell_anno_fn, sep = '\t', header = False, index = False)

### Subset matrix

In [17]:
adata.X = adata.X.toarray()
adata_new = adata[adata.obs['cell'].isin(new_cells), :].copy()
adata_new

AnnData object with n_obs × n_vars = 1600 × 33538
    obs: 'cell'
    var: 'feature_id', 'feature_name'

In [18]:
save_10x_data(
    adata_new, 
    out_dir = out_matrix_dir,
    layer = None,
    row_is_cell = True,
    cell_columns = None, feature_columns = None, barcode_columns = None           
)

In [19]:
if os.path.exists(os.path.join(out_matrix_dir, 'cell_anno.tsv')) and os.path.exists(os.path.join(out_matrix_dir, 'spot_anno.tsv')):
    os.remove(os.path.join(out_matrix_dir, 'cell_anno.tsv'))